In [1]:
"""
Phase 2 — Significance Tests (Paired per-arm controls)
Each treatment arm tested against its OWN per-arm control group.
"""
import warnings
import pandas as pd
import statsmodels.api as sm
warnings.simplefilter("ignore")

INPUT_PATH = "Output_phase_2/output_customer_level_2026-03-30.csv"
OUTPUT_PATH = "Output/phase2_significance_results_2026-03-30.xlsx"


def load_data(path: str = INPUT_PATH) -> pd.DataFrame:
    df = pd.read_csv(path)
    split = df["churn_group"].str.split(" - ", n=1, expand=True)
    raw_model = split[0].str.strip()
    df["incentive"] = split[1].str.strip()
    df["is_control"] = raw_model.str.contains("control", case=False)
    df["base_model"] = raw_model.str.replace(r"\s*control\s*:?\s*", "", regex=True).str.strip()
    df["reactivated"] = (df["reactivated"] > 0).astype(int)
    return df


def _one_sided(model_cls, y_ctrl, y_trt):
    y = pd.concat([y_ctrl, y_trt])
    x = sm.add_constant([0] * len(y_ctrl) + [1] * len(y_trt))
    mod = model_cls(y, x).fit(disp=0) if model_cls == sm.Logit else model_cls(y, x).fit()
    coef, p2 = mod.params.iloc[1], mod.pvalues.iloc[1]
    p = p2 / 2 if coef > 0 else 1.0
    stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
    return coef, round(p, 4), stars


def _build_row(model, inc, c, t):
    _, p_r, s_r = _one_sided(sm.Logit, c["reactivated"], t["reactivated"])
    _, p_s, s_s = _one_sided(sm.OLS, c["total_sales"], t["total_sales"])
    _, p_m, s_m = _one_sided(sm.OLS, c["margin"], t["margin"])
    return {
        "Model": model, "Incentive": inc,
        "N_Ctrl": len(c), "N_Treat": len(t),
        "React_Ctrl": round(c["reactivated"].mean(), 4),
        "React_Treat": round(t["reactivated"].mean(), 4),
        "Uplift_React": round(t["reactivated"].mean() - c["reactivated"].mean(), 4),
        "p_react": p_r, "sig_react": s_r,
        "Sales_Ctrl": round(c["total_sales"].mean(), 4),
        "Sales_Treat": round(t["total_sales"].mean(), 4),
        "Uplift_Sales": round(t["total_sales"].mean() - c["total_sales"].mean(), 4),
        "p_sales": p_s, "sig_sales": s_s,
        "Margin_Ctrl": round(c["margin"].mean(), 4),
        "Margin_Treat": round(t["margin"].mean(), 4),
        "Uplift_Margin": round(t["margin"].mean() - c["margin"].mean(), 4),
        "p_margin": p_m, "sig_margin": s_m,
    }


def test_pooled(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for model, mdf in df.groupby("base_model"):
        c, t = mdf[mdf["is_control"]], mdf[~mdf["is_control"]]
        if c.empty or t.empty:
            continue
        rows.append(_build_row(model, "ALL (pooled)", c, t))
    return pd.DataFrame(rows).sort_values("p_react").reset_index(drop=True)


def test_per_arm(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (model, inc), gdf in df.groupby(["base_model", "incentive"]):
        c, t = gdf[gdf["is_control"]], gdf[~gdf["is_control"]]
        if c.empty or t.empty:
            continue
        rows.append(_build_row(model, inc, c, t))
    return pd.DataFrame(rows).sort_values(["Model", "p_react"]).reset_index(drop=True)


if __name__ == "__main__":
    df = load_data()
    print(f"Loaded {len(df):,} rows | Models: {df['base_model'].unique().tolist()}")

    pooled = test_pooled(df)
    per_arm = test_per_arm(df)

    with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as w:
        pooled.to_excel(w, sheet_name="Pooled", index=False)
        per_arm.to_excel(w, sheet_name="Per Arm", index=False)

    print("\n=== POOLED ===")
    print(pooled.to_string(index=False))
    print("\n=== PER ARM ===")
    print(per_arm.to_string(index=False))
    print(f"\nSaved to {OUTPUT_PATH}")

Loaded 70,165 rows | Models: ['Binary Uplift', 'MTUM']

=== POOLED ===
        Model    Incentive  N_Ctrl  N_Treat  React_Ctrl  React_Treat  Uplift_React  p_react sig_react  Sales_Ctrl  Sales_Treat  Uplift_Sales  p_sales sig_sales  Margin_Ctrl  Margin_Treat  Uplift_Margin  p_margin sig_margin
         MTUM ALL (pooled)   18221    16720      0.0117       0.0151        0.0033   0.0035        **      0.2990       0.4127        0.1137   0.0034        **       0.1801        0.2456         0.0655    0.0055         **
Binary Uplift ALL (pooled)   18376    16848      0.0114       0.0119        0.0005   0.3317                0.3547       0.3489       -0.0058   1.0000                 0.2194        0.2130        -0.0065    1.0000           

=== PER ARM ===
        Model                        Incentive  N_Ctrl  N_Treat  React_Ctrl  React_Treat  Uplift_React  p_react sig_react  Sales_Ctrl  Sales_Treat  Uplift_Sales  p_sales sig_sales  Margin_Ctrl  Margin_Treat  Uplift_Margin  p_margin sig_margin
